In [ ]:
!pip -q install transformers[torch] datasets accelerate sentencepiece sacrebleu

# 🚀 Dyslexia-Friendly Restructuring Trainer (V9 - IRON-CLAD VERSION)

### 📋 Instructions:
1. **STRICT RESTRUCTURING**: Added a command to prevent the AI from being 'lazy' and copying the input.
2. Ensure **Runtime > Change runtime type** is set to **T4 GPU**.
3. Run the cells in order.

In [ ]:
!pip -q install transformers[torch] datasets accelerate sentencepiece sacrebleu

In [ ]:
from google.colab import files
import os

print("Please upload 'train.json' and 'val.json'.")
uploaded = files.upload()

In [ ]:
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

MODEL_NAME = "google/flan-t5-large"
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 128

def load_local_data(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    # IRON-CLAD PROMPT: Forces restructuring and prevents copying
    formatted = [{
        "input_text": f"Instruction: Rewrite and simplify the following text into exactly 4 short, simple sentences. DO NOT COPY THE INPUT WORD-FOR-WORD. Use very simple language for a dyslexic child.\nLanguage: {item['language']}\nInput: {item['input']}\nOutput:", 
        "target_text": item['output']
    } for item in data]
    return Dataset.from_list(formatted)

train_ds = load_local_data("train.json")
val_ds = load_local_data("val.json")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.gradient_checkpointing_enable()

def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=MAX_INPUT_LENGTH, truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples["target_text"], max_length=MAX_TARGET_LENGTH, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_val = val_ds.map(preprocess_function, batched=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5, 
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=32, 
    per_device_eval_batch_size=1,
    warmup_steps=100,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,    
    optim="adafactor",             
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("🚀 Starting Iron-Clad Training (FLAN-T5-LARGE)...")
trainer.train()
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")
print("✅ Final Training Complete!")

In [ ]:
def predict(text, lang="en"):
    prompt = f"Instruction: Rewrite and simplify the following text into exactly 4 short, simple sentences. DO NOT COPY THE INPUT WORD-FOR-WORD. Use very simple language for a dyslexic child.\nLanguage: {lang}\nInput: {text}\nOutput:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, 
        max_length=MAX_TARGET_LENGTH, 
        num_beams=4, 
        no_repeat_ngram_size=3,
        repetition_penalty=2.0
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- TEST RUN ---")
test_en = "Photosynthesis is the process used by plants and other organisms to convert light energy into chemical energy."
print(f"English: {predict(test_en, 'en')}")

test_fil = "Ang fotosintesis ay ang proseso kung saan ginagamit ng mga halaman ang sikat ng araw upang gumawa ng pagkain."
print(f"Filipino: {predict(test_fil, 'fil')}")